In [ ]:
# ROGII Wellbore Geology Prediction - v12 submission notebook
#
# v12 = v11 + Triple-Signal's two particle filters.
#
# Architecture (6 spatial signals -> LGB x 5 + XGB -> ridge -> shrinkage -> EWM):
#   1. Spatial plane fit KNN  (konbu17)
#   2. Row KNN K=20           (konbu17, n_q=8000)
#   3. MLP+PE-L8 multi-out, 3-seed ensemble (v9 -> v11)
#   4. Anisotropic-exponential kriging (v11)
#   5. TVT-PF, Z-velocity coupled (v12 NEW)
#   6. ANCC-PF, S = TVT + Z tracker (v12 NEW)
#
# The two PFs are run in PARALLEL via multiprocessing.Pool over wells
# (0.15 sec/well measured on M1 Pro 10-core; expect ~3 min for 773 wells
# on Kaggle 4-core).

import os
import sys
import base64
import json
import logging
import time
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
)
logger = logging.getLogger("rogii.v12")

# 1) Modules
SRC_DIR = "/kaggle/working/rogii_src"
os.makedirs(SRC_DIR, exist_ok=True)

_MODULES = {
    "feature_builder.py": "...",
    "neural_ancc.py": "IiIiTmV1cmFsIEFOQ0MoWCwgWSkgc3VyZmFjZSBtb2RlbC4KCkh5cG90aGVzaXMgKGZyb20gU1RSQVRFR1lfUkVTRVQpOiBrb25idTE3J3MgcGVyLXdlbGwgcGxhbmUgZml0IGFuZCByb3ctbGV2ZWwKS05OIGFyZSAqbG9jYWwqIOKAlCB0aGV5IGZhaWwgaW4gc3BhdGlhbCByZWdpb25zIHdpdGggc3BhcnNlIHRyYWluaW5nIG5laWdoYm9ycywKcHJvZHVjaW5nIHRoZSBjYXRhc3Ryb3BoaWMgd2VsbC1STVNFIG91dGxpZXJzIHdlIHNlZSBpbiB2OCBPT0YgKG1heCA1NiBmdCkuCkEgc21hbGwgTUxQIHdpdGggc2ludXNvaWRhbCBwb3NpdGlvbmFsIGVuY29kaW5nIChOZVJGLXN0eWxlKSBvbiAoWCwgWSkgbGVhcm5zCmEgKmdsb2JhbCBzbW9vdGggc3VyZmFjZSogdGhhdCBzaG91bGQgZ2VuZXJhbGl6ZSBiZXR0ZXIgaW4gdGhvc2Ugc3BhcnNlIHJlZ2lvbnMKd2hpbGUgc3RpbGwgbWF0Y2hpbmcgbG9jYWwgZmVhdHVyZXMgdmlhIHRoZSBoaWdoLWZyZXF1ZW5jeSBQRSBiYW5kcy4KClRoZSBsb2FkLWJlYXJpbmcgaWRlbnRpdHk6CiAgICBUVlQgPSAtWiArIEFOQ0MgKyBiX3dlbGwgICAoaW50cmEtd2VsbCBzaWdtYSAwLjAwNjUgZnQsIGV4YWN0KQoKU28gcHJlZGljdGluZyBBTkNDKFgsIFkpIGF0IGEgaGVsZC1vdXQgd2VsbCdzIChYLCBZKSwgdGhlbiBwbHVnZ2luZyBpbnRvIHRoZQpjbG9zZWQtZm9ybSBUVlQgZm9ybXVsYSB3aXRoIHRoZSB3ZWxsJ3MgbWVkaWFuIGJfd2VsbCBmcm9tIGl0cyB2aXNpYmxlIHByZWZpeCwKaXMgc3VmZmljaWVudCB0byByZWNvdmVyIFRWVCB3aXRoIHRoZSBzYW1lIGZpZGVsaXR5IGFzIEFOQ0MuCgpEZXNpZ24gKHBlciB0aGUgYnJpZWYsIG5vIHR1bmluZyBzYWdhKToKICAxLiAoWCwgWSkgbm9ybWFsaXplZCB0byBbLTEsIDFdIG92ZXIgdGhlIHRyYWluIGV4dGVudC4KICAyLiBTaW51c29pZGFsIHBvc2l0aW9uYWwgZW5jb2Rpbmc6IGdhbW1hKHApID0gW3NpbigyXmsgKiBwaSAqIHApLCBjb3MoLi4uKV0KICAgICBmb3IgayA9IDAuLkwtMSwgYXBwbGllZCB0byBYIGFuZCBZIHNlcGFyYXRlbHkuIE91dHB1dCBkaW0gPSA0ICogTCBwZXIgY29vcmQuCiAgICAgUGx1cyB0aGUgcmF3IChYLCBZKSBmZWF0dXJlIGNvbmNhdGVuYXRlZC4KICAzLiBNTFA6IDQgbGF5ZXJzIHggMjU2LCBTaUxVLCBOZVJGLXN0eWxlIHNraXAgZnJvbSBpbnB1dCB0byBsYXllciAyLgogIDQuIEFkYW0sIGxyPTFlLTMsIGNvc2luZSBkZWNheSwgYmF0Y2ggNDA5NiwgNTAwayByb3dzIC8gZXBvY2gsIE1QUyBiYWNrZW5kLgogIDUuIFNxdWFyZWQgbG9zcyBvbiBBTkNDLiBNdWx0aS1vdXRwdXQgdmFyaWFudCBwcmVkaWN0cyBhbGwgNiBmb3JtYXRpb24gdG9wcy4KCkFsbCB0cmFpbmluZyBpcyBwZXItZm9sZCAodHJhaW4gZm9sZCByb3dzIG9ubHkpLiBJbmZlcmVuY2U6IG1vZGVsLnByZWRpY3RfeHkKb24gdGhlIGhlbGQtb3V0IChYLCBZKSAtPiBBTkNDIC0+ICgtWiArIEFOQ0MgKyBiX3ByZWZpeF9tZWRpYW4pIC0+IFRWVC4KClRoaXMgbW9kdWxlIGlzIHNlbGYtY29udGFpbmVkIOKAlCBubyBwYW5kYXMsIHBvbGFycyArIHRvcmNoIG9ubHkuIFRoZSA1TSAoWCwgWSwKZm9ybWF0aW9uKSB0dXBsZXMgZml0IGNvbWZvcnRhYmx5IGluIDMyR0IuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgSXRlcmFibGUsIFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBvbGFycyBhcyBwbAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgpGT1JNQVRJT05TOiB0dXBsZVtzdHIsIC4uLl0gPSAoIkFOQ0MiLCAiQVNUTlUiLCAiQVNUTkwiLCAiRUdGRFUiLCAiRUdGREwiLCAiQlVEQSIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBEYXRhIGxvYWRpbmcKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmRlZiBsb2FkX3RyYWluX3Jvd3MoCiAgICB0cmFpbl9kaXI6IFBhdGgsCiAgICBmb3JtYXRpb25zOiBTZXF1ZW5jZVtzdHJdID0gRk9STUFUSU9OUywKICAgIHBhdGhzOiBJdGVyYWJsZVtQYXRoXSB8IE5vbmUgPSBOb25lLAopIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgIiIiTG9hZCBhbGwgKFgsIFksIGZvcm1hdGlvbnNbLCB3ZWxsXSkgcm93cyBmcm9tIHRyYWluaW5nIENTVnMuCgogICAgUmV0dXJucwogICAgLS0tLS0tLQogICAgeHkgOiAoTiwgMikgZmxvYXQ2NAogICAgdGFyZ2V0cyA6IChOLCBGKSBmbG9hdDMyICAgRiA9IGxlbihmb3JtYXRpb25zKQogICAgd2lkcyA6IChOLCkgb2JqZWN0IHN0ciAgICB3ZWxsIElEIHBlciByb3cKICAgICIiIgogICAgaWYgcGF0aHMgaXMgTm9uZToKICAgICAgICBwYXRocyA9IHNvcnRlZChQYXRoKHRyYWluX2RpcikuZ2xvYigiKl9faG9yaXpvbnRhbF93ZWxsLmNzdiIpKQogICAgY29scyA9IFsiWCIsICJZIiwgKmZvcm1hdGlvbnNdCiAgICB4eV9ibG9ja3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgZl9ibG9ja3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgd2lkX2Jsb2NrczogbGlzdFtucC5uZGFycmF5XSA9IFtdCiAgICBza2lwcGVkID0gMAogICAgZm9yIHAgaW4gcGF0aHM6CiAgICAgICAgd2lkID0gcC5zdGVtLnJlcGxhY2UoIl9faG9yaXpvbnRhbF93ZWxsIiwgIiIpCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIEZvcmNlIEFOQ0MgZmxvYXQgKHNvbWUgd2VsbHMgc3RvcmUgaXQgYXMgVXRmOCB3aXRoIGFsbC1udWxsKTsKICAgICAgICAgICAgIyBwb2xhcnMgcmVhZF9jc3Ygc2NoZW1hX292ZXJyaWRlcyBoYW5kbGVzIGVpdGhlci4KICAgICAgICAgICAgZGYgPSBwbC5yZWFkX2NzdihwLCBjb2x1bW5zPWNvbHMsIGluZmVyX3NjaGVtYV9sZW5ndGg9MTBfMDAwKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHNraXBwZWQgKz0gMQogICAgICAgICAgICBjb250aW51ZQogICAgICAgICMgQ29lcmNlIGFsbCBmb3JtYXRpb25zIHRvIEZsb2F0NjQgaWYgdGhleSBjYW1lIGJhY2sgYXMgVXRmOC4KICAgICAgICBmb3IgYyBpbiBmb3JtYXRpb25zOgogICAgICAgICAgICBpZiBkZltjXS5kdHlwZSA9PSBwbC5VdGY4OgogICAgICAgICAgICAgICAgZGYgPSBkZi53aXRoX2NvbHVtbnMocGwuY29sKGMpLmNhc3QocGwuRmxvYXQ2NCwgc3RyaWN0PUZhbHNlKSkKICAgICAgICBkZiA9IGRmLmRyb3BfbnVsbHMoc3Vic2V0PWNvbHMpCiAgICAgICAgaWYgbGVuKGRmKSA9PSAwOgogICAgICAgICAgICBza2lwcGVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICB4eV9ibG9ja3MuYXBwZW5kKGRmLnNlbGVjdChbIlgiLCAiWSJdKS50b19udW1weSgpLmFzdHlwZShucC5mbG9hdDY0KSkKICAgICAgICBmX2Jsb2Nrcy5hcHBlbmQoZGYuc2VsZWN0KGxpc3QoZm9ybWF0aW9ucykpLnRvX251bXB5KCkuYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgICAgIHdpZF9ibG9ja3MuYXBwZW5kKG5wLmZ1bGwobGVuKGRmKSwgd2lkLCBkdHlwZT1vYmplY3QpKQogICAgaWYgbm90IHh5X2Jsb2NrczoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJObyB0cmFpbmluZyByb3dzIGxvYWRlZCBmcm9tIHt0cmFpbl9kaXJ9IikKICAgIHh5ID0gbnAuY29uY2F0ZW5hdGUoeHlfYmxvY2tzKQogICAgdGFyZ2V0cyA9IG5wLmNvbmNhdGVuYXRlKGZfYmxvY2tzKQogICAgd2lkcyA9IG5wLmNvbmNhdGVuYXRlKHdpZF9ibG9ja3MpCiAgICByZXR1cm4geHksIHRhcmdldHMsIHdpZHMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIE5ldXJhbCBtb2RlbAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKY2xhc3MgUG9zaXRpb25hbEVuY29kaW5nKG5uLk1vZHVsZSk6CiAgICAiIiJOZVJGLXN0eWxlIHNpbnVzb2lkYWwgcG9zaXRpb25hbCBlbmNvZGluZyBvbiBlYWNoIGNvb3JkaW5hdGUuCgogICAgcCBhc3N1bWVkIG5vcm1hbGl6ZWQgdG8gcm91Z2hseSBbLTEsIDFdLiBPdXRwdXQgZGltID0gNCpMIChjb3Mvc2luIHggMiBjb29yZHMpLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9mcmVxczogaW50KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm51bV9mcmVxcyA9IG51bV9mcmVxcwogICAgICAgICMgMl5rICogcGkgZm9yIGsgPSAwIC4uIEwtMQogICAgICAgIGZyZXFzID0gKDIuMCAqKiB0b3JjaC5hcmFuZ2UobnVtX2ZyZXFzKSkgKiBtYXRoLnBpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImZyZXFzIiwgZnJlcXMudG8odG9yY2guZmxvYXQzMikpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeHk6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICMgeHk6IChCLCAyKQogICAgICAgIGlmIHNlbGYubnVtX2ZyZXFzID09IDA6CiAgICAgICAgICAgIHJldHVybiB4eQogICAgICAgIHNjYWxlZCA9IHh5LnVuc3F1ZWV6ZSgtMSkgKiBzZWxmLmZyZXFzICAgIyAoQiwgMiwgTCkKICAgICAgICBzaW4gPSB0b3JjaC5zaW4oc2NhbGVkKQogICAgICAgIGNvcyA9IHRvcmNoLmNvcyhzY2FsZWQpCiAgICAgICAgIyBpbnRlcmxlYXZlIHRvIChCLCA0ICogTCkKICAgICAgICBlbmNvZGVkID0gdG9yY2guY2F0KFtzaW4sIGNvc10sIGRpbT0tMSkuZmxhdHRlbihzdGFydF9kaW09MSkKICAgICAgICByZXR1cm4gdG9yY2guY2F0KFt4eSwgZW5jb2RlZF0sIGRpbT0tMSkgICMgcmF3ICsgUEUKCgpjbGFzcyBOZXJmTUxQKG5uLk1vZHVsZSk6CiAgICAiIiI0IGhpZGRlbiBsYXllcnMgeCAyNTYsIFNpTFUsIHdpdGggc2tpcCBmcm9tIGlucHV0IHRvIGxheWVyIDIuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2RpbTogaW50LCBoaWRkZW46IGludCwgb3V0X2RpbTogaW50KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmZjMSA9IG5uLkxpbmVhcihpbl9kaW0sIGhpZGRlbikKICAgICAgICBzZWxmLmZjMiA9IG5uLkxpbmVhcihoaWRkZW4sIGhpZGRlbikKICAgICAgICBzZWxmLmZjMyA9IG5uLkxpbmVhcihoaWRkZW4gKyBpbl9kaW0sIGhpZGRlbikgICAjIHNraXAKICAgICAgICBzZWxmLmZjNCA9IG5uLkxpbmVhcihoaWRkZW4sIGhpZGRlbikKICAgICAgICBzZWxmLmhlYWQgPSBubi5MaW5lYXIoaGlkZGVuLCBvdXRfZGltKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGggPSBGLnNpbHUoc2VsZi5mYzEoeCkpCiAgICAgICAgaCA9IEYuc2lsdShzZWxmLmZjMihoKSkKICAgICAgICBoID0gRi5zaWx1KHNlbGYuZmMzKHRvcmNoLmNhdChbaCwgeF0sIGRpbT0tMSkpKQogICAgICAgIGggPSBGLnNpbHUoc2VsZi5mYzQoaCkpCiAgICAgICAgcmV0dXJuIHNlbGYuaGVhZChoKQoKCkBkYXRhY2xhc3MKY2xhc3MgVHJhaW5Db25maWc6CiAgICBudW1fZnJlcXM6IGludCA9IDgKICAgIGhpZGRlbjogaW50ID0gMjU2CiAgICBvdXRfZGltOiBpbnQgPSAxCiAgICByb3dzX3Blcl9lcG9jaDogaW50ID0gNTAwXzAwMAogICAgYmF0Y2hfc2l6ZTogaW50ID0gNDA5NgogICAgZXBvY2hzOiBpbnQgPSAxMgogICAgbHI6IGZsb2F0ID0gMWUtMwogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDAuMAogICAgc2VlZDogaW50ID0gNDIKICAgIGRldmljZTogc3RyID0gIm1wcyIgaWYgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIKICAgIHZhbF9mcmFjOiBmbG9hdCA9IDAuMCAgIyBubyBpbnRlcm5hbCB2YWw6IGV4dGVybmFsIEdyb3VwS0ZvbGQgb3ducyB2YWwuCgoKY2xhc3MgQW5jY05ldDoKICAgICIiIldyYXBzIHRoZSBtb2RlbCArIHRyYWluIGV4dGVudCBub3JtYWxpemVyICsgdHJhaW4gbG9vcC4KCiAgICBvdXRfZGltPT0xIC0+IEFOQ0Mgb25seS4gb3V0X2RpbT09bGVuKEZPUk1BVElPTlMpIC0+IGFsbC1mb3JtYXRpb25zIGhlYWQuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2ZnOiBUcmFpbkNvbmZpZyk6CiAgICAgICAgc2VsZi5jZmcgPSBjZmcKICAgICAgICB0b3JjaC5tYW51YWxfc2VlZChjZmcuc2VlZCkKICAgICAgICBzZWxmLnBlID0gUG9zaXRpb25hbEVuY29kaW5nKGNmZy5udW1fZnJlcXMpCiAgICAgICAgaW5fZGltID0gMiArICg0ICogY2ZnLm51bV9mcmVxcyBpZiBjZmcubnVtX2ZyZXFzID4gMCBlbHNlIDApCiAgICAgICAgc2VsZi5tbHAgPSBOZXJmTUxQKGluX2RpbSwgY2ZnLmhpZGRlbiwgY2ZnLm91dF9kaW0pCiAgICAgICAgc2VsZi5kZXZpY2UgPSB0b3JjaC5kZXZpY2UoY2ZnLmRldmljZSkKICAgICAgICBzZWxmLnBlLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHNlbGYubWxwLnRvKHNlbGYuZGV2aWNlKQogICAgICAgICMgdHJhaW4tZXh0ZW50IG5vcm1hbGl6ZXIgKHNldCBpbiBmaXQpCiAgICAgICAgc2VsZi54X21pZCA9IDAuMAogICAgICAgIHNlbGYueF9zY2wgPSAxLjAKICAgICAgICBzZWxmLnlfbWlkID0gMC4wCiAgICAgICAgc2VsZi55X3NjbCA9IDEuMAogICAgICAgICMgdGFyZ2V0IG5vcm1hbGl6ZXIgKG1lYW4gLyBzdGQgcGVyIG91dHB1dCBkaW0sIHNldCBpbiBmaXQpCiAgICAgICAgc2VsZi50X21lYW4gPSBucC56ZXJvcyhjZmcub3V0X2RpbSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBzZWxmLnRfc3RkID0gbnAub25lcyhjZmcub3V0X2RpbSwgZHR5cGU9bnAuZmxvYXQzMikKCiAgICAjIC0tIG5vcm1hbGl6ZXJzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgogICAgZGVmIF9maXRfbm9ybShzZWxmLCB4eTogbnAubmRhcnJheSwgdGFyZ2V0czogbnAubmRhcnJheSkgLT4gTm9uZToKICAgICAgICB4X21pbiwgeF9tYXggPSBmbG9hdCh4eVs6LCAwXS5taW4oKSksIGZsb2F0KHh5WzosIDBdLm1heCgpKQogICAgICAgIHlfbWluLCB5X21heCA9IGZsb2F0KHh5WzosIDFdLm1pbigpKSwgZmxvYXQoeHlbOiwgMV0ubWF4KCkpCiAgICAgICAgc2VsZi54X21pZCA9IDAuNSAqICh4X21pbiArIHhfbWF4KQogICAgICAgIHNlbGYueF9zY2wgPSBtYXgoMC41ICogKHhfbWF4IC0geF9taW4pLCAxLjApCiAgICAgICAgc2VsZi55X21pZCA9IDAuNSAqICh5X21pbiArIHlfbWF4KQogICAgICAgIHNlbGYueV9zY2wgPSBtYXgoMC41ICogKHlfbWF4IC0geV9taW4pLCAxLjApCiAgICAgICAgc2VsZi50X21lYW4gPSB0YXJnZXRzLm1lYW4oYXhpcz0wKS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICAjIEF2b2lkIGRpdi1ieS16ZXJvIG9uIGRlZ2VuZXJhdGUgY2FzZXMuCiAgICAgICAgc2VsZi50X3N0ZCA9IG5wLm1heGltdW0odGFyZ2V0cy5zdGQoYXhpcz0wKSwgMS4wKS5hc3R5cGUobnAuZmxvYXQzMikKCiAgICBkZWYgX25vcm1feHkoc2VsZiwgeHk6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgb3V0ID0gbnAuZW1wdHlfbGlrZSh4eSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBvdXRbOiwgMF0gPSAoeHlbOiwgMF0gLSBzZWxmLnhfbWlkKSAvIHNlbGYueF9zY2wKICAgICAgICBvdXRbOiwgMV0gPSAoeHlbOiwgMV0gLSBzZWxmLnlfbWlkKSAvIHNlbGYueV9zY2wKICAgICAgICByZXR1cm4gb3V0CgogICAgIyAtLSB0cmFpbmluZyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgogICAgZGVmIGZpdChzZWxmLCB4eV90cmFpbjogbnAubmRhcnJheSwgdF90cmFpbjogbnAubmRhcnJheSwgKiwgdmVyYm9zZTogYm9vbCA9IEZhbHNlKSAtPiBkaWN0OgogICAgICAgIGNmZyA9IHNlbGYuY2ZnCiAgICAgICAgaWYgdF90cmFpbi5uZGltID09IDE6CiAgICAgICAgICAgIHRfdHJhaW4gPSB0X3RyYWluLnJlc2hhcGUoLTEsIDEpCiAgICAgICAgYXNzZXJ0IHRfdHJhaW4uc2hhcGVbMV0gPT0gY2ZnLm91dF9kaW0sICh0X3RyYWluLnNoYXBlLCBjZmcub3V0X2RpbSkKICAgICAgICBzZWxmLl9maXRfbm9ybSh4eV90cmFpbiwgdF90cmFpbikKICAgICAgICB4eV9uID0gc2VsZi5fbm9ybV94eSh4eV90cmFpbikKICAgICAgICB0X24gPSAoKHRfdHJhaW4gLSBzZWxmLnRfbWVhbikgLyBzZWxmLnRfc3RkKS5hc3R5cGUobnAuZmxvYXQzMikKCiAgICAgICAgZGV2aWNlID0gc2VsZi5kZXZpY2UKICAgICAgICB4eV90ID0gdG9yY2guZnJvbV9udW1weSh4eV9uKS50byhkZXZpY2UpCiAgICAgICAgdF90ID0gdG9yY2guZnJvbV9udW1weSh0X24pLnRvKGRldmljZSkKICAgICAgICBOID0geHlfdC5zaGFwZVswXQoKICAgICAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtKHNlbGYubWxwLnBhcmFtZXRlcnMoKSwgbHI9Y2ZnLmxyLCB3ZWlnaHRfZGVjYXk9Y2ZnLndlaWdodF9kZWNheSkKICAgICAgICAjIENvc2luZSBkZWNheSBvdmVyIHRvdGFsIGl0ZXJhdGlvbnMgKGFjcm9zcyBhbGwgZXBvY2hzKS4KICAgICAgICByb3dzX3Blcl9lcG9jaCA9IG1pbihjZmcucm93c19wZXJfZXBvY2gsIE4pCiAgICAgICAgc3RlcHNfcGVyX2Vwb2NoID0gbWF4KHJvd3NfcGVyX2Vwb2NoIC8vIGNmZy5iYXRjaF9zaXplLCAxKQogICAgICAgIHRvdGFsX3N0ZXBzID0gY2ZnLmVwb2NocyAqIHN0ZXBzX3Blcl9lcG9jaAogICAgICAgIHNjaGVkID0gdG9yY2gub3B0aW0ubHJfc2NoZWR1bGVyLkNvc2luZUFubmVhbGluZ0xSKG9wdCwgVF9tYXg9dG90YWxfc3RlcHMpCgogICAgICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhjZmcuc2VlZCkKICAgICAgICBlcG9jaF9sb3NzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgdF9zdGFydCA9IHRpbWUucGVyZl9jb3VudGVyKCkKCiAgICAgICAgc2VsZi5tbHAudHJhaW4oKQogICAgICAgIGZvciBlcCBpbiByYW5nZShjZmcuZXBvY2hzKToKICAgICAgICAgICAgIyBTYW1wbGUgcm93c19wZXJfZXBvY2ggcmFuZG9tIHJvdyBpbmRpY2VzIGZvciB0aGlzIGVwb2NoLgogICAgICAgICAgICBzZWwgPSB0b3JjaC5mcm9tX251bXB5KAogICAgICAgICAgICAgICAgcm5nLmNob2ljZShOLCByb3dzX3Blcl9lcG9jaCwgcmVwbGFjZT1GYWxzZSkuYXN0eXBlKG5wLmludDY0KQogICAgICAgICAgICApLnRvKGRldmljZSkKICAgICAgICAgICAgeHlfZXAgPSB4eV90W3NlbF0KICAgICAgICAgICAgdF9lcCA9IHRfdFtzZWxdCiAgICAgICAgICAgICMgU2h1ZmZsZSB3aXRoaW4gZXBvY2ggaXMgaW1wbGljaXQgYnkgc2FtcGxpbmcuCiAgICAgICAgICAgIG5fbG9zcyA9IDAuMAogICAgICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCByb3dzX3Blcl9lcG9jaCwgY2ZnLmJhdGNoX3NpemUpOgogICAgICAgICAgICAgICAgeGIgPSB4eV9lcFtzOnMgKyBjZmcuYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgICAgIHliID0gdF9lcFtzOnMgKyBjZmcuYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgICAgIGZlYXRzID0gc2VsZi5wZSh4YikKICAgICAgICAgICAgICAgIHByZWQgPSBzZWxmLm1scChmZWF0cykKICAgICAgICAgICAgICAgIGxvc3MgPSBGLm1zZV9sb3NzKHByZWQsIHliKQogICAgICAgICAgICAgICAgb3B0Lnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgICAgICBvcHQuc3RlcCgpCiAgICAgICAgICAgICAgICBzY2hlZC5zdGVwKCkKICAgICAgICAgICAgICAgIG5fbG9zcyArPSBsb3NzLml0ZW0oKSAqIHhiLnNoYXBlWzBdCiAgICAgICAgICAgIGF2ZyA9IG5fbG9zcyAvIHJvd3NfcGVyX2Vwb2NoCiAgICAgICAgICAgIGVwb2NoX2xvc3MuYXBwZW5kKGF2ZykKICAgICAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgICAgIHByaW50KGYiICBlcCB7ZXA6MDJkfSAgbG9zcyhub3JtKT17YXZnOi41Zn0gIGxyPXtvcHQucGFyYW1fZ3JvdXBzWzBdWydsciddOi4yZX0iLCBmbHVzaD1UcnVlKQogICAgICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdF9zdGFydAogICAgICAgIHJldHVybiB7ImVwb2NoX2xvc3MiOiBlcG9jaF9sb3NzLCAiZml0X3RpbWVfcyI6IGVsYXBzZWR9CgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHByZWRpY3Qoc2VsZiwgeHk6IG5wLm5kYXJyYXksICosIGJhdGNoX3NpemU6IGludCA9IDIwMF8wMDApIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiUHJlZGljdCB0YXJnZXRzIGF0IHh5LiBSZXR1cm5zIChOLCBvdXRfZGltKSBmbG9hdDMyIGluIG9yaWdpbmFsIHVuaXRzLiIiIgogICAgICAgIHNlbGYubWxwLmV2YWwoKQogICAgICAgIHh5X24gPSBzZWxmLl9ub3JtX3h5KHh5KQogICAgICAgIHh5X3QgPSB0b3JjaC5mcm9tX251bXB5KHh5X24pLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIG91dHM6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgICAgIGZvciBzIGluIHJhbmdlKDAsIHh5X3Quc2hhcGVbMF0sIGJhdGNoX3NpemUpOgogICAgICAgICAgICBmZWF0cyA9IHNlbGYucGUoeHlfdFtzOnMgKyBiYXRjaF9zaXplXSkKICAgICAgICAgICAgcHJlZCA9IHNlbGYubWxwKGZlYXRzKS5jcHUoKS5udW1weSgpCiAgICAgICAgICAgIG91dHMuYXBwZW5kKHByZWQpCiAgICAgICAgb3V0ID0gbnAuY29uY2F0ZW5hdGUob3V0cywgYXhpcz0wKQogICAgICAgIG91dCA9IG91dCAqIHNlbGYudF9zdGRbTm9uZSwgOl0gKyBzZWxmLnRfbWVhbltOb25lLCA6XQogICAgICAgIHJldHVybiBvdXQuYXN0eXBlKG5wLmZsb2F0MzIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBDbG9zZWQtZm9ybSBiX3dlbGwgZnJvbSBwcmVmaXgKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmRlZiBmaXRfYl9wcmVmaXhfbWVkaWFuKAogICAgcHJlZml4X3R2dF9pbnB1dDogbnAubmRhcnJheSwgcHJlZml4X3o6IG5wLm5kYXJyYXksIHByZWZpeF9hbmNjX3ByZWQ6IG5wLm5kYXJyYXkKKSAtPiBmbG9hdDoKICAgICIiIk1lZGlhbiBwZXItcm93IGIgPSBUVlRfaW5wdXQgKyBaIC0gQU5DQ19wcmVkIG92ZXIgdGhlIHZpc2libGUgcHJlZml4LiIiIgogICAgcmVzID0gcHJlZml4X3R2dF9pbnB1dCArIHByZWZpeF96IC0gcHJlZml4X2FuY2NfcHJlZAogICAgcmVzID0gcmVzW25wLmlzZmluaXRlKHJlcyldCiAgICByZXR1cm4gZmxvYXQobnAubWVkaWFuKHJlcykpIGlmIHJlcy5zaXplIGVsc2UgMC4wCg==",
    "aniso_kriging.py": "IiIiQW5pc290cm9waWMgbG9jYWwga3JpZ2luZyBmb3IgZm9ybWF0aW9uLXRvcCBzdXJmYWNlcy4KCmtvbmJ1MTcgdXNlcyBpc290cm9waWMgSURXICh3aXRoIFgvWSBzdGQtc2NhbGluZykgb24gYW4gfjVNLXJvdyBzcGF0aWFsCmdyaWQuIFRoZSBhdWRpdCBhdHRyaWJ1dGVzIDAuMy0wLjYgUk1TRSBwb3RlbnRpYWwgdG8gcmVwbGFjaW5nIHRoaXMgd2l0aAphbmlzb3Ryb3BpYyBrcmlnaW5nIHRoYXQgcmVzcGVjdHMgdGhlIHJlZ2lvbmFsIE5FLVNXIEVhZ2xlIEZvcmQgc3RyaWtlLgoKVGhpcyBtb2R1bGUgaXMgdGhlIHY5IHN0YXJ0aW5nIHBvaW50LiBJdCBpcyAqKm5vdCB3aXJlZCBpbnRvIHY4Kio7IHRoZSB2OQpiZW5jaCB3aWxsIHBsdWcgaXQgaW4gYnkgcmVwbGFjaW5nIGBgUm93S05OLmltcHV0ZWBgIGluIGZlYXR1cmVfYnVpbGRlci4KCkRlc2lnbiBkZWNpc2lvbnM6CiAgKiBBbmlzb3Ryb3B5IHZpYSBhIDJ4MiBTUEQgd2hpdGVuaW5nIG1hdHJpeCBXIGFwcGxpZWQgdG8gKFgsIFkpIGJlZm9yZQogICAga2R0cmVlIGxvb2t1cC4gVyBpcyBlc3RpbWF0ZWQgZW1waXJpY2FsbHkgZnJvbSB0aGUgQU5DQyBncmFkaWVudAogICAgZmllbGQgb2YgdHJhaW4gZGF0YSAoUENBIG9uIChkQU5DQy9kWCwgZEFOQ0MvZFkpKSBvciBzZXQgZXhwbGljaXRseQogICAgYnkgdXNlci4gVGhlICJsb25nIiBheGlzIG9mIFcgaXMgdGhlIHN0YWJsZS1kaXJlY3Rpb24gKGFsb25nIHN0cmlrZSkuCiAgKiBMb2NhbCBrcmlnaW5nIHdlaWdodHM6IEdhdXNzaWFuIGtlcm5lbAogICAgICAgIHdfaSA9IGV4cCgtIDAuNSAqICgoeF9pIC0geF9xKV5UIFdeVCBXICh4X2kgLSB4X3EpKSApCiAgICAod2l0aCBhIHNtYWxsIHJpZGdlIHRvIGtlZXAgdGhlIGtyaWdpbmcgbWF0cml4IHdlbGwtY29uZGl0aW9uZWQpLgogICogUHJlZGljdGl2ZSB2YXJpYW5jZSBpcyBleHBvc2VkIGZvciB0aGUgR0JNIHRvIHVzZSBhcyBhIGZlYXR1cmUuCgpDb21wdXRlIGJ1ZGdldDogc3RpbGwgTyhLKSBwZXIgcXVlcnkgYWZ0ZXIga2R0cmVlIG5hcnJvd2luZywgc28gdGhlCm92ZXJoZWFkIHZzIElEVyBpcyB0aW55LgoKUmVmZXJlbmNlIChmcmVlIG9mIGphcmdvbik6CiAgICBGb3IgYSBzdGF0aW9uYXJ5LCBub3JtYWxseS1kaXN0cmlidXRlZCBzdXJmYWNlIHRoZSBvcHRpbWFsIGxpbmVhcgogICAgZXN0aW1hdG9yIGF0IGEgcXVlcnkgcG9pbnQgaXMgYSB3ZWlnaHRlZCBhdmVyYWdlIG9mIG5lYXJieSBzYW1wbGVzCiAgICB3aGVyZSB0aGUgd2VpZ2h0cyBzb2x2ZSB0aGUga3JpZ2luZyBzeXN0ZW0KICAgICAgICBLIHcgPSBrX3EKICAgIEtfaWogPSBjb3ZhcmlhbmNlKHhfaSwgeF9qKSwgIGtfcSxpID0gY292YXJpYW5jZSh4X3EsIHhfaSkuCiAgICBXaXRoIGEgR2F1c3NpYW4vZXhwb25lbnRpYWwga2VybmVsICsgYSB0aW55IHJpZGdlIChudWdnZXQpLCB0aGlzIGlzCiAgICBhIDIweDIwIGxpbmVhciBzb2x2ZSBwZXIgcXVlcnkgYXQgSz0yMCDigJQgZXNzZW50aWFsbHkgZnJlZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNjaXB5LnNwYXRpYWwgaW1wb3J0IGNLRFRyZWUKCgpkZWYgZXN0aW1hdGVfYW5pc290cm9weV9mcm9tX2ZpZWxkKAogICAgeHk6IG5wLm5kYXJyYXksCiAgICB6OiBucC5uZGFycmF5LAogICAgKiwKICAgIHNpZ21hX3JhdGlvOiBmbG9hdCA9IDMuMCwKICAgIGVwczogZmxvYXQgPSAxZS05LAopIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgIiIiRXN0aW1hdGUgYW5pc290cm9weSBheGlzICYgbGVuZ3RoIHNjYWxlcyBmcm9tIGEgbm9pc3kgc3BhdGlhbCBmaWVsZC4KCiAgICBBcHByb2FjaDogZml0IGEgZ2xvYmFsIGxpbmVhciB0cmVuZCBgYHogfiBhICsgYnggKyBjeWBgIG9uIGEgc3BhcnNlLAogICAgd2VsbC1hd2FyZSBzdWJzYW1wbGUuIFRoZSBncmFkaWVudCBkaXJlY3Rpb24gKGIsIGMpIGdpdmVzIHRoZSBkaXAKICAgIGRpcmVjdGlvbiAoaGlnaC12YXJpYXRpb24gYXhpcykuIFRoZSBzdHJpa2UgZGlyZWN0aW9uIGlzIHBlcnBlbmRpY3VsYXIuCiAgICBMZW5ndGgtc2NhbGUgcmF0aW8gaXMgc2V0IGJ5IGBgc2lnbWFfcmF0aW9gYCBzaW5jZSBvbmUtc2hvdCBlc3RpbWF0aW9uCiAgICBvZiByYXRpbyBmcm9tIGRhdGEgaXMgYnJpdHRsZS4KCiAgICBXaHkgbm90IGVzdGltYXRlIHRoZSByYXRpbyBmcm9tIGRhdGE/IEluIEVhZ2xlIEZvcmQsIHRoZSAqZGlyZWN0aW9uKgogICAgb2Ygc3RyaWtlIGlzIHdlbGwtZGVmaW5lZCBieSB0aGUgcmVnaW9uYWwgZGlwICh+TkUtU1cgc3RyaWtlIGZyb20gYQogICAgU0UtZGlwcGluZyBzaGVsZiksIGJ1dCB0aGUgKnJhdGlvKiBvZiBhbG9uZy1zdHJpa2UgdnMgY3Jvc3Mtc3RyaWtlCiAgICBjb3JyZWxhdGlvbiBsZW5ndGggaXMgYSB0dW5hYmxlIGtyaWdpbmcgcGFyYW1ldGVyLCBub3QgYSBmaXhlZCBzdXJmYWNlCiAgICBwcm9wZXJ0eS4gV2UgZGVmYXVsdCB0byAzOjEg4oCUIGVtcGlyaWNhbGx5IG5lYXIgdGhlIG9wdGltdW0gb24gdGhlCiAgICBFYWdsZSBGb3JkIGNvbXBldGl0aW9uIHN1cmZhY2UgKHN3ZWVwIHRlc3RlZCBhdCAyMDAgd2VsbHM6IHJhdGlvcyBvZgogICAgMi00IGdpdmUgUk1TRSAxMC0xNSwgcmF0aW9zIG9mIDEgb3IgMTAgZ2l2ZSAzMCspLgoKICAgIFBhcmFtZXRlcnMKICAgIC0tLS0tLS0tLS0KICAgIHh5IDogKE4sIDIpIGZsb2F0NjQKICAgIHogIDogKE4sKSAgIGZsb2F0NjQgICAgICAgc2FtcGxlZCB2YWx1ZXMgb2YgdGhlIHN1cmZhY2UgYXQgeHkKICAgIHNpZ21hX3JhdGlvIDogYWxvbmctc3RyaWtlIDogY3Jvc3Mtc3RyaWtlIGNvcnJlbGF0aW9uIGxlbmd0aCByYXRpby4KCiAgICBSZXR1cm5zCiAgICAtLS0tLS0tCiAgICBSIDogKDIsIDIpIHJvdGF0aW9uIG1hdHJpeDsgY29sdW1ucyA9IChoaWdoLWdyYWRpZW50LCBhbG9uZy1zdHJpa2UpIGF4ZXMKICAgIHNpZ21hIDogKDIsKSA9IFsxLjAsIHNpZ21hX3JhdGlvXQogICAgIiIiCiAgICBpZiB4eS5zaGFwZVswXSAhPSB6LnNoYXBlWzBdIG9yIHh5LnNoYXBlWzFdICE9IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigieHkgbXVzdCBiZSAoTiwyKSwgeiBtdXN0IGJlIChOLCkiKQogICAgaWYgeHkuc2hhcGVbMF0gPCAyMDoKICAgICAgICByZXR1cm4gbnAuZXllKDIpLCBucC5hcnJheShbMS4wLCAxLjBdKQoKICAgICMgU3Vic2FtcGxlIHNvIGRlbnNlIGludHJhLXdlbGwgcm93cyBkbyBub3QgZG9taW5hdGUgdGhlIExTIGZpdAogICAgbl9zdWIgPSBtaW4oMjAwXzAwMCwgeHkuc2hhcGVbMF0pCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMjAyNjA1MDcpCiAgICBpZHhfc3ViID0gcm5nLmNob2ljZSh4eS5zaGFwZVswXSwgbl9zdWIsIHJlcGxhY2U9RmFsc2UpCiAgICBBID0gbnAuY29sdW1uX3N0YWNrKFtucC5vbmVzKG5fc3ViKSwgeHlbaWR4X3N1YiwgMF0sIHh5W2lkeF9zdWIsIDFdXSkKICAgIGNvZWYsICpfID0gbnAubGluYWxnLmxzdHNxKEEsIHpbaWR4X3N1Yl0sIHJjb25kPU5vbmUpCiAgICBncmFkID0gY29lZlsxOjNdCiAgICBncmFkX25vcm0gPSBmbG9hdChucC5saW5hbGcubm9ybShncmFkKSkKICAgIGlmIGdyYWRfbm9ybSA8IGVwczoKICAgICAgICByZXR1cm4gbnAuZXllKDIpLCBucC5hcnJheShbMS4wLCAxLjBdKQoKICAgICMgSGlnaC1ncmFkaWVudCAoZGlwKSBkaXJlY3Rpb24gPSB1bml0IHZlY3RvciBhbG9uZyBncmFkCiAgICBkaXAgPSBncmFkIC8gZ3JhZF9ub3JtICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgKDIsKQogICAgIyBTdHJpa2UgZGlyZWN0aW9uID0gcGVycGVuZGljdWxhciAocm90YXRlIDkwwrApCiAgICBzdHJpa2UgPSBucC5hcnJheShbLWRpcFsxXSwgZGlwWzBdXSkKICAgIFIgPSBucC5jb2x1bW5fc3RhY2soW2RpcCwgc3RyaWtlXSkgICAgICAgICAgICAgICAjICgyLCAyKQogICAgc2lnbWEgPSBucC5hcnJheShbMS4wLCBmbG9hdChzaWdtYV9yYXRpbyldKQogICAgcmV0dXJuIFIsIHNpZ21hCgoKQGRhdGFjbGFzcwpjbGFzcyBBbmlzb0Zvcm1hdGlvbktOTjoKICAgICIiIkFuaXNvdHJvcGljIGxvY2FsIGtyaWdpbmcgcHJlZGljdG9yIGZvciBvbmUgZm9ybWF0aW9uIHRvcC4KCiAgICBCdWlsZCBvbmNlIG9uIGFsbCB0cmFpbiByb3dzOyBxdWVyeSBwZXIgdGVzdCByb3cuCgogICAgTm90ZXMKICAgIC0tLS0tCiAgICBUaGUgd2hpdGVuaW5nIG1hdHJpeCBMID0gUiBAIGRpYWcoMSAvIChzaWdtYSAqIHJhbmdlX3NjYWxlICogTF9ub3JtKSkuCiAgICBMX25vcm0gaXMgYW4gb3ZlcmFsbCBsZW5ndGggc2NhbGUgKG1lZGlhbiBuZWFyZXN0LW5laWdoYm9yIGRpc3RhbmNlIGluCiAgICByYXcgd2hpdGVuZWQgc3BhY2UpIHRoYXQgZW5zdXJlcyB0aGUga2VybmVsIGFyZ3VtZW50IGlzIE8oMSkgYXQgdHlwaWNhbAogICAgaW50ZXItcm93IGRpc3RhbmNlcy4gcmFuZ2Vfc2NhbGUgZnVydGhlciB0aWdodGVucyAoPDEpIG9yIGxvb3NlbnMgKD4xKQogICAgdGhlIGtlcm5lbCBkZWNheS4KICAgICIiIgoKICAgIHh5OiBucC5uZGFycmF5ICAgICAgICAgICAgICAjIChOLCAyKSBvcmlnaW5hbCBjb29yZHMKICAgIHo6IG5wLm5kYXJyYXkgICAgICAgICAgICAgICAjIChOLCkgdGFyZ2V0IHZhbHVlcwogICAgd2VsbF9pZHM6IG5wLm5kYXJyYXkgICAgICAgICMgKE4sKSBpbnRlZ2VyIHdlbGwtaWRzCiAgICB3ZWxsX2luZGV4OiBsaXN0W3N0cl0KICAgIFI6IG5wLm5kYXJyYXkgICAgICAgICAgICAgICAjICgyLCAyKSByb3RhdGlvbgogICAgc2lnbWE6IG5wLm5kYXJyYXkgICAgICAgICAgICMgKDIsKSBsZW5ndGggc2NhbGVzCiAgICBMOiBucC5uZGFycmF5ICAgICAgICAgICAgICAgIyAoMiwgMikgd2hpdGVuaW5nID0gUiAvIChzaWdtYSAqIHJhbmdlX3NjYWxlICogTF9ub3JtKQogICAgTF9ub3JtOiBmbG9hdCAgICAgICAgICAgICAgICMgb3ZlcmFsbCBsZW5ndGggc2NhbGUKICAgIHRyZWU6IGNLRFRyZWUKICAgIG51Z2dldDogZmxvYXQKICAgIHJhbmdlX3NjYWxlOiBmbG9hdCAgICAgICAgICAjIG11bHRpcGxpZXIgZm9yIHNpZ21hIC0+IGtyaWdpbmcgbGVuZ3RoCgogICAgZGVmIHdlbGxfdG9faW50KHNlbGYsIHdpZDogc3RyKSAtPiBpbnQ6CiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4gc2VsZi53ZWxsX2luZGV4LmluZGV4KHdpZCkKICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgcmV0dXJuIC0xCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgZml0KAogICAgICAgIGNscywKICAgICAgICB4eTogbnAubmRhcnJheSwKICAgICAgICB6OiBucC5uZGFycmF5LAogICAgICAgIHdlbGxfaWRzOiBucC5uZGFycmF5LAogICAgICAgIHdlbGxfaW5kZXg6IGxpc3Rbc3RyXSwKICAgICAgICAqLAogICAgICAgIGFuaXNvdHJvcHk6IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgbnVnZ2V0OiBmbG9hdCA9IDFlLTQsCiAgICAgICAgcmFuZ2Vfc2NhbGU6IGZsb2F0ID0gMS4wLAogICAgKSAtPiAiQW5pc29Gb3JtYXRpb25LTk4iOgogICAgICAgIGlmIGFuaXNvdHJvcHkgaXMgTm9uZToKICAgICAgICAgICAgUiwgc2lnbWEgPSBlc3RpbWF0ZV9hbmlzb3Ryb3B5X2Zyb21fZmllbGQoeHksIHopCiAgICAgICAgZWxzZToKICAgICAgICAgICAgUiwgc2lnbWEgPSBhbmlzb3Ryb3B5CgogICAgICAgICMgRmlyc3QtcGFzcyB3aGl0ZW5pbmcgKGp1c3Qgcm90YXRpb24gKyBzaWdtYSk6IHVzZWQgdG8gbGVhcm4gTF9ub3JtCiAgICAgICAgIyBzbyBrZXJuZWwgYXJndW1lbnQgaXMgTygxKSBhdCB0eXBpY2FsIG5laWdoYm9yIGRpc3RhbmNlLgogICAgICAgIExfcHJlID0gUiBAIG5wLmRpYWcoMS4wIC8gc2lnbWEpCiAgICAgICAgeHlfcHJlID0geHkgQCBMX3ByZQoKICAgICAgICAjIFNldCBMX25vcm0gdG8gdGhlIGludGVyLXdlbGwgbGVuZ3RoIHNjYWxlLCBOT1QgdGhlIGludHJhLXdlbGwgcm93CiAgICAgICAgIyBzcGFjaW5nLiBFYWNoIHdlbGwgaGFzIHRob3VzYW5kcyBvZiBkZW5zZSByb3dzIGFsb25nIGl0cyB0cmFjaywKICAgICAgICAjIHNvICJtZWRpYW4gTk4gb3ZlciBhbGwgcm93cyIgaXMgbWlzbGVhZGluZ2x5IHNtYWxsLiBUaGUgcmVsZXZhbnQKICAgICAgICAjIHNjYWxlIGZvciBoZWxkLW91dCBxdWVyaWVzIGlzIHRoZSBtZWRpYW4gZGlzdGFuY2UgZnJvbSBvbmUgd2VsbAogICAgICAgICMgQ0VOVFJPSUQgdG8gaXRzIG5lYXJlc3Qgb3RoZXIgd2VsbCBjZW50cm9pZCBpbiB0aGUgcm90YXRlZCBmcmFtZS4KICAgICAgICAjIFRoaXMgaXMgb24gdGhlIG9yZGVyIG9mIHR5cGljYWwgd2VsbCBzcGFjaW5nLiBXZSBjb21wdXRlIGNlbnRyb2lkcwogICAgICAgICMgdmlhIGEgYmluY291bnQtYmFzZWQgZ3JvdXAtbWVhbiB0byBhdm9pZCBhbiBPKE4gKiBXKSBkb3VibGUgbG9vcC4KICAgICAgICB1bmlxdWVfd2lkcywgaW52ID0gbnAudW5pcXVlKHdlbGxfaWRzLCByZXR1cm5faW52ZXJzZT1UcnVlKQogICAgICAgIGlmIGxlbih1bmlxdWVfd2lkcykgPj0gNDoKICAgICAgICAgICAgY291bnRzID0gbnAuYmluY291bnQoaW52KS5hc3R5cGUobnAuZmxvYXQ2NCkKICAgICAgICAgICAgc3Vtc194ID0gbnAuYmluY291bnQoaW52LCB3ZWlnaHRzPXh5X3ByZVs6LCAwXSkKICAgICAgICAgICAgc3Vtc195ID0gbnAuYmluY291bnQoaW52LCB3ZWlnaHRzPXh5X3ByZVs6LCAxXSkKICAgICAgICAgICAgY2VudHJvaWRzID0gbnAuY29sdW1uX3N0YWNrKFtzdW1zX3ggLyBjb3VudHMsIHN1bXNfeSAvIGNvdW50c10pCiAgICAgICAgICAgIHRyZWVfYyA9IGNLRFRyZWUoY2VudHJvaWRzKQogICAgICAgICAgICBkX2MsIF8gPSB0cmVlX2MucXVlcnkoY2VudHJvaWRzLCBrPTIpCiAgICAgICAgICAgIExfbm9ybSA9IGZsb2F0KG5wLm1lZGlhbihkX2NbOiwgMV0pKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGJib3hfbWluID0geHlfcHJlLm1pbihheGlzPTApCiAgICAgICAgICAgIGJib3hfbWF4ID0geHlfcHJlLm1heChheGlzPTApCiAgICAgICAgICAgIGJib3hfc3BhbiA9IGZsb2F0KG5wLm1heGltdW0oYmJveF9tYXggLSBiYm94X21pbiwgMS4wKS5tZWFuKCkpCiAgICAgICAgICAgIExfbm9ybSA9IGJib3hfc3BhbiAvIDMwLjAKICAgICAgICBMX25vcm0gPSBtYXgoTF9ub3JtLCAxZS05KQoKICAgICAgICAjIEZpbmFsIHdoaXRlbmluZzogcm90YXRlLCBhbmlzb3Ryb3B5LXNjYWxlLCB0aGVuIGRpdmlkZSBieSBvdmVyYWxsCiAgICAgICAgIyBsZW5ndGggc2NhbGUgKiByYW5nZV9zY2FsZS4KICAgICAgICAjIEwgPSBMX3ByZSAvIChMX25vcm0gKiByYW5nZV9zY2FsZSkgc28ga2VybmVsIGFyZyB+MSBuZWFyIHR5cGljYWwgTk4uCiAgICAgICAgTCA9IExfcHJlIC8gKExfbm9ybSAqIHJhbmdlX3NjYWxlKQogICAgICAgIHh5X3doaXRlID0geHkgQCBMCiAgICAgICAgdHJlZSA9IGNLRFRyZWUoeHlfd2hpdGUpCiAgICAgICAgcmV0dXJuIGNscyh4eT14eSwgej16LCB3ZWxsX2lkcz13ZWxsX2lkcywgd2VsbF9pbmRleD13ZWxsX2luZGV4LAogICAgICAgICAgICAgICAgICAgUj1SLCBzaWdtYT1zaWdtYSwgTD1MLCBMX25vcm09TF9ub3JtLCB0cmVlPXRyZWUsCiAgICAgICAgICAgICAgICAgICBudWdnZXQ9bnVnZ2V0LCByYW5nZV9zY2FsZT1yYW5nZV9zY2FsZSkKCiAgICBkZWYgcXVlcnkoCiAgICAgICAgc2VsZiwKICAgICAgICB4eV9xOiBucC5uZGFycmF5LAogICAgICAgICosCiAgICAgICAgazogaW50ID0gMjAsCiAgICAgICAga2VybmVsOiBzdHIgPSAiZ2F1c3NpYW4iLCAgICMgImdhdXNzaWFuIiB8ICJleHBvbmVudGlhbCIKICAgICAgICBiYXRjaF9zaXplOiBpbnQgPSAyMDBfMDAwLAogICAgKSAtPiB0dXBsZVtucC5uZGFycmF5LCBucC5uZGFycmF5LCBucC5uZGFycmF5XToKICAgICAgICAiIiJSZXR1cm5zIChtZWFuX3ByZWQsIHN0ZF9wcmVkLCBtaW5fZGlzdCkuCgogICAgICAgIFNlbGYtd2VsbCBleGNsdXNpb24gaXMgaW50ZW50aW9uYWxseSBOT1QgZG9uZSBoZXJlLiBGb3IgYmVuY2htYXJrLwogICAgICAgIE9PRiB1c2UsIHRoZSBjYWxsZXIgaXMgZXhwZWN0ZWQgdG8gaGF2ZSBidWlsdCB0aGlzIG9iamVjdCB3aXRoIG9ubHkKICAgICAgICB0aGUgdHJhaW4tZm9sZCByb3dzLCBzbyBsZWFrYWdlIGlzIGltcG9zc2libGUgYnkgY29uc3RydWN0aW9uLgogICAgICAgICIiIgogICAgICAgIGlmIGtlcm5lbCBub3QgaW4gKCJnYXVzc2lhbiIsICJleHBvbmVudGlhbCIpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBrZXJuZWwge2tlcm5lbCFyfSIpCiAgICAgICAgbiA9IHh5X3Euc2hhcGVbMF0KICAgICAgICBtZWFucyA9IG5wLmZ1bGwobiwgbnAubmFuLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgICAgIHN0ZHMgPSBucC5mdWxsKG4sIG5wLm5hbiwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICBtaW5fZGlzdCA9IG5wLmZ1bGwobiwgbnAuaW5mLCBkdHlwZT1ucC5mbG9hdDY0KQoKICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgbiwgYmF0Y2hfc2l6ZSk6CiAgICAgICAgICAgIHN0b3AgPSBtaW4oc3RhcnQgKyBiYXRjaF9zaXplLCBuKQogICAgICAgICAgICB4eV9iID0geHlfcVtzdGFydDpzdG9wXQogICAgICAgICAgICBxX3doaXRlID0geHlfYiBAIHNlbGYuTAogICAgICAgICAgICBkX2ssIGlkeF9rID0gc2VsZi50cmVlLnRyZWUucXVlcnkocV93aGl0ZSwgaz1rLCB3b3JrZXJzPS0xKSBcCiAgICAgICAgICAgICAgICBpZiBGYWxzZSBlbHNlIHNlbGYudHJlZS5xdWVyeShxX3doaXRlLCBrPWssIHdvcmtlcnM9LTEpCiAgICAgICAgICAgICMgY0tEVHJlZSByZXR1cm5zIChCLCkgYXJyYXlzIGZvciBrPTE7IGVuc3VyZSAyLUQuCiAgICAgICAgICAgIGlmIGRfay5uZGltID09IDE6CiAgICAgICAgICAgICAgICBkX2sgPSBkX2tbOiwgTm9uZV0KICAgICAgICAgICAgICAgIGlkeF9rID0gaWR4X2tbOiwgTm9uZV0KICAgICAgICAgICAgdmFsaWRfayA9IG5wLmlzZmluaXRlKGRfaykKICAgICAgICAgICAgbWluX2Rpc3Rbc3RhcnQ6c3RvcF0gPSBucC53aGVyZSh2YWxpZF9rLCBkX2ssIG5wLmluZikubWluKGF4aXM9MSkKCiAgICAgICAgICAgIGlmIGtlcm5lbCA9PSAiZ2F1c3NpYW4iOgogICAgICAgICAgICAgICAgY19pID0gbnAud2hlcmUodmFsaWRfaywgbnAuZXhwKC0wLjUgKiBkX2sgKiBkX2spLCAwLjApCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBjX2kgPSBucC53aGVyZSh2YWxpZF9rLCBucC5leHAoLWRfayksIDAuMCkKCiAgICAgICAgICAgICMgQmF0Y2hlZCBrcmlnaW5nIHN5c3RlbS4gQnVpbGQgKEIsIEssIEspIEdyYW0gbWF0cml4IGZyb20gbmVpZ2hib3IKICAgICAgICAgICAgIyB3aGl0ZW5lZCBjb29yZHMgdXNpbmcgdGhlIHNxdWFyZWQtbm9ybSBpZGVudGl0eToKICAgICAgICAgICAgIyAgIGReMihpLGopID0gfHhfaXxeMiArIHx4X2p8XjIgLSAyIHhfaS54X2oKICAgICAgICAgICAgIyB0byBhdm9pZCB0aGUgKEIsIEssIEssIDIpIGludGVybWVkaWF0ZS4KICAgICAgICAgICAgeHlfbiA9IHNlbGYueHlbaWR4X2tdIEAgc2VsZi5MICAgICAgICAgICAgICAgICMgKEIsIEssIDIpCiAgICAgICAgICAgIHNxX24gPSAoeHlfbiAqIHh5X24pLnN1bShheGlzPS0xKSAgICAgICAgICAgICAjIChCLCBLKQogICAgICAgICAgICBkb3QgPSBucC5laW5zdW0oImJpZCxiamQtPmJpaiIsIHh5X24sIHh5X24pICAgIyAoQiwgSywgSykKICAgICAgICAgICAgZDIgPSBzcV9uWzosIDosIE5vbmVdICsgc3Ffbls6LCBOb25lLCA6XSAtIDIuMCAqIGRvdAogICAgICAgICAgICBucC5tYXhpbXVtKGQyLCAwLjAsIG91dD1kMikKICAgICAgICAgICAgaWYga2VybmVsID09ICJnYXVzc2lhbiI6CiAgICAgICAgICAgICAgICBLX21hdCA9IG5wLmV4cCgtMC41ICogZDIpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBLX21hdCA9IG5wLmV4cCgtbnAuc3FydChkMikpCiAgICAgICAgICAgIEtfbWF0ID0gS19tYXQgKyBzZWxmLm51Z2dldCAqIG5wLmV5ZShrKVtOb25lLCA6LCA6XQoKICAgICAgICAgICAgIyBTb2x2ZSBLX21hdFtpXSBAIHdbaV0gPSBjX2lbaV0gIChCIHN5c3RlbXMgb2Ygc2l6ZSBLKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB3ID0gbnAubGluYWxnLnNvbHZlKEtfbWF0LCBjX2lbLi4uLCBOb25lXSkuc3F1ZWV6ZSgtMSkKICAgICAgICAgICAgZXhjZXB0IG5wLmxpbmFsZy5MaW5BbGdFcnJvcjoKICAgICAgICAgICAgICAgIHcgPSBucC5mdWxsX2xpa2UoY19pLCBucC5uYW4pCgogICAgICAgICAgICAjIE51bWVyaWNhbGx5LWRlZ2VuZXJhdGUgcm93czogd2VpZ2h0cyBhbGwgc3ViLVVMUCBvciBub24tZmluaXRlLgogICAgICAgICAgICAjIEZhbGwgYmFjayB0byBJRFc7IGlmIGV2ZW4gSURXIHJvdy1zdW0gaXMgdGlueSwgdXNlIHVuaWZvcm0gMS9LLgogICAgICAgICAgICB3c3VtID0gdy5zdW0oYXhpcz0xKQogICAgICAgICAgICBiYWRfc29sdmUgPSAofm5wLmlzZmluaXRlKHcpLmFsbChheGlzPTEpKSB8IChucC5hYnMod3N1bSkgPCAxZS0xMikKICAgICAgICAgICAgaWYgYmFkX3NvbHZlLmFueSgpOgogICAgICAgICAgICAgICAgcm93X3N1bSA9IGNfaS5zdW0oYXhpcz0xLCBrZWVwZGltcz1UcnVlKQogICAgICAgICAgICAgICAgdGlueSA9IHJvd19zdW0gPCAxZS0xMgogICAgICAgICAgICAgICAgcm93X3N1bV9zYWZlID0gbnAud2hlcmUodGlueSwgMS4wLCByb3dfc3VtKQogICAgICAgICAgICAgICAgd19mYWxsYmFjayA9IG5wLndoZXJlKHRpbnksIDEuMCAvIGssIGNfaSAvIHJvd19zdW1fc2FmZSkKICAgICAgICAgICAgICAgIHcgPSBucC53aGVyZShiYWRfc29sdmVbOiwgTm9uZV0sIHdfZmFsbGJhY2ssIHcpCiAgICAgICAgICAgICAgICB3c3VtID0gdy5zdW0oYXhpcz0xKQoKICAgICAgICAgICAgd3N1bV9zYWZlID0gbnAud2hlcmUobnAuYWJzKHdzdW0pIDwgMWUtMTIsIDEuMCwgd3N1bSkKICAgICAgICAgICAgel9uID0gc2VsZi56W2lkeF9rXSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgKEIsIEspCiAgICAgICAgICAgIG1lYW5zX2IgPSAoel9uICogdykuc3VtKGF4aXM9MSkgLyB3c3VtX3NhZmUKICAgICAgICAgICAgIyBWYXJpYW5jZTogMSAtIGMuVCBAIHcgIChjbGlwcGVkKQogICAgICAgICAgICB2YXJfYiA9IG5wLmNsaXAoMS4wIC0gKGNfaSAqIHcpLnN1bShheGlzPTEpLCAwLjAsIE5vbmUpCiAgICAgICAgICAgIHN0ZF9iID0gbnAuc3FydCh2YXJfYikKCiAgICAgICAgICAgIG5vX25laWdoID0gfm5wLmFueSh2YWxpZF9rLCBheGlzPTEpCiAgICAgICAgICAgIG1lYW5zX2IgPSBucC53aGVyZShub19uZWlnaCwgbnAubmFuLCBtZWFuc19iKQogICAgICAgICAgICBzdGRfYiA9IG5wLndoZXJlKG5vX25laWdoLCBucC5uYW4sIHN0ZF9iKQogICAgICAgICAgICBtZWFuc1tzdGFydDpzdG9wXSA9IG1lYW5zX2IKICAgICAgICAgICAgc3Rkc1tzdGFydDpzdG9wXSA9IHN0ZF9iCgogICAgICAgIHJldHVybiBtZWFucywgc3RkcywgbWluX2Rpc3QKCgpkZWYgZml0X2FuaXNvX2Zvcl9mb3JtYXRpb25zKAogICAgdHJhaW5fcGF0aHM6IGxpc3RbUGF0aF0sCiAgICBmb3JtYXRpb25zOiB0dXBsZVtzdHIsIC4uLl0gPSAoIkFOQ0MiLCAiQVNUTlUiLCAiQVNUTkwiLCAiRUdGRFUiLCAiRUdGREwiLCAiQlVEQSIpLAogICAgKiwKICAgIHJhbmdlX3NjYWxlOiBmbG9hdCA9IDEuMCwKKSAtPiBkaWN0W3N0ciwgQW5pc29Gb3JtYXRpb25LTk5dOgogICAgIiIiQnVpbGQgb25lIEFuaXNvRm9ybWF0aW9uS05OIHBlciBmb3JtYXRpb24uIFRoZSBhbmlzb3Ryb3B5IGRpcmVjdGlvbgogICAgaXMgZXN0aW1hdGVkIGluZGVwZW5kZW50bHkgcGVyIGZvcm1hdGlvbjsgaW4gcHJhY3RpY2UgdGhleSBzaG91bGQgYmUKICAgIHNpbWlsYXIgZm9yIHBhcmFsbGVsIGZvcm1hdGlvbiB0b3BzLgogICAgIiIiCiAgICBjb2xzID0gWyJYIiwgIlkiLCAqZm9ybWF0aW9uc10KICAgIHhzLCB5cyA9IFtdLCBbXQogICAgZl9hcnJzOiBsaXN0W25wLm5kYXJyYXldID0gW10KICAgIHdpZF9hcnI6IGxpc3Rbc3RyXSA9IFtdCiAgICBmb3IgcCBpbiB0cmFpbl9wYXRoczoKICAgICAgICB3aWQgPSBwLnN0ZW0ucmVwbGFjZSgiX19ob3Jpem9udGFsX3dlbGwiLCAiIikKICAgICAgICB0cnk6CiAgICAgICAgICAgIGRmID0gcGQucmVhZF9jc3YocCwgdXNlY29scz1jb2xzKS5kcm9wbmEoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgZGYuZW1wdHk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgeHMuYXBwZW5kKGRmWyJYIl0udG9fbnVtcHkoKSkKICAgICAgICB5cy5hcHBlbmQoZGZbIlkiXS50b19udW1weSgpKQogICAgICAgIGZfYXJycy5hcHBlbmQoZGZbbGlzdChmb3JtYXRpb25zKV0udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkpCiAgICAgICAgd2lkX2Fyci5leHRlbmQoW3dpZF0gKiBsZW4oZGYpKQoKICAgIHh5ID0gbnAuY29sdW1uX3N0YWNrKFtucC5jb25jYXRlbmF0ZSh4cyksIG5wLmNvbmNhdGVuYXRlKHlzKV0pCiAgICBmX3RhcmdldHMgPSBucC52c3RhY2soZl9hcnJzKQogICAgd2VsbF9pbmRleCA9IHNvcnRlZChzZXQod2lkX2FycikpCiAgICB3ZWxsX3BvcyA9IHt3OiBpIGZvciBpLCB3IGluIGVudW1lcmF0ZSh3ZWxsX2luZGV4KX0KICAgIHdlbGxfaWRzID0gbnAuYXJyYXkoW3dlbGxfcG9zW3ddIGZvciB3IGluIHdpZF9hcnJdLCBkdHlwZT1ucC5pbnQzMikKCiAgICBvdXQ6IGRpY3Rbc3RyLCBBbmlzb0Zvcm1hdGlvbktOTl0gPSB7fQogICAgZm9yIGosIGZuYW1lIGluIGVudW1lcmF0ZShmb3JtYXRpb25zKToKICAgICAgICB6ID0gZl90YXJnZXRzWzosIGpdCiAgICAgICAgb3V0W2ZuYW1lXSA9IEFuaXNvRm9ybWF0aW9uS05OLmZpdCgKICAgICAgICAgICAgeHksIHosIHdlbGxfaWRzLCB3ZWxsX2luZGV4LAogICAgICAgICAgICByYW5nZV9zY2FsZT1yYW5nZV9zY2FsZSwKICAgICAgICApCiAgICByZXR1cm4gb3V0Cg==",
    "anchor_shrinkage.py": "IiIiQW5jaG9yLXNocmlua2FnZSBwb3N0LXByb2Nlc3NvciBmb3IgdjEwLgoKVGhlIG91dGxpZXIgZGlhZ25vc2lzIG9uIHY5IChPT0YgbWF4IHdlbGwgUk1TRSAxNjUsIDgvMTEgY2F0YXN0cm9waGljCndlbGxzID0gTU9ERUwgRFJJRlQsIG5vdCByZWFsIGdlb3N0ZWVyaW5nIGFjdGlvbikgc2hvd3MgdGhhdCBmb3IgdGhlc2UKd2VsbHMgYSB0cml2aWFsIGJhc2VsaW5lIC0tICJwcmVkaWN0IHRoZSByb2xsaW5nIG1lYW4gb2YgdGhlIGxhc3QgMTAwCnByZWZpeCBUVlRfaW5wdXQgdmFsdWVzIGZvciBldmVyeSBldmFsIHJvdyIgLS0gd291bGQgc2NvcmUgNS0xNSBmdApSTVNFIHZlcnN1cyBvdXIgbW9kZWwncyA1My0xNjYgZnQuIFRoZSBtb2RlbCBpcyAqb3Zlci1jb25maWRlbnRseQptb3ZpbmcqIHByZWRpY3Rpb25zIGF3YXkgZnJvbSB0aGUgYW5jaG9yIHdoZW4gdGhlcmUncyBubyBnZW9sb2dpY2FsCnJlYXNvbiB0by4KClRoaXMgcG9zdC1wcm9jZXNzb3IgYmxlbmRzIHRoZSBtb2RlbCdzIHByZWRpY3RlZCBkZWx0YSB3aXRoIHplcm8KKHplcm8gPSAic3RheSBhdCB0aGUgYW5jaG9yIiwgc2luY2UgdGFyZ2V0ID0gVFZUIC0gbGFzdF9rbm93bl9UVlQpOgoKICAgIGRlbHRhX2ZpbmFsID0gYWxwaGEgKiBkZWx0YV9tb2RlbAogICAgVFZUX2ZpbmFsICA9IGxhc3Rfa25vd25fVFZUICsgZGVsdGFfZmluYWwKCndpdGggYWxwaGEgPCAxLiBKYW1lcy1TdGVpbi1zdHlsZSBtdWx0aXBsaWNhdGl2ZSBzaHJpbmthZ2UuCgpUaGUgb3B0aW1hbCBhbHBoYSBiYWxhbmNlczoKICAtIFRoZSBjb3N0IG9mIHNocmlua2luZyBHT09EIHByZWRpY3Rpb25zIChyZWR1Y2VzIHNpZ25hbCBvbiByZWFsLQogICAgbW90aW9uIHdlbGxzIHdoZXJlIHRhcmdldCBpcyBub24tdHJpdmlhbCkuCiAgLSBUaGUgYmVuZWZpdCBvZiBkYW1waW5nIENBVEFTVFJPUEhJQyBwcmVkaWN0aW9ucyAod2hlcmUgdGFyZ2V0IGlzCiAgICBuZWFyIHplcm8gYnV0IG1vZGVsIHNheXMgwrE1MGZ0KS4KCkVtcGlyaWNhbGx5IGNhbGlicmF0ZWQgYWdhaW5zdCBwb3B1bGF0aW9uIGJhc2VsaW5lcyAobWVkaWFuIGV2YWwtCm9mZnNldCAwLjg0IGZ0LCBwOTUgMjAuNCwgcDk5IDM3LjcpOiBzaHJpbmtpbmcgYnkgMC42LTAuOCBzaG91bGQKcmVkdWNlIGNhdGFzdHJvcGhpYyBtYXgtd2VsbC1STVNFIHN1YnN0YW50aWFsbHkgd2hpbGUgbG9zaW5nIGxpdHRsZQpvbiB0aGUgdHlwaWNhbCBtZWRpYW4uCgpBIG1vcmUgc29waGlzdGljYXRlZCB2YXJpYW50IChgZ2F0ZWRfc2hyaW5rYWdlYCkgdXNlcyBhIHBlci1yb3cKY29uZmlkZW5jZSBzaWduYWwgLS0gS05OIG5laWdoYm9yIGRpc3RhbmNlLCBNTFAtdnMtS05OIGRpc2FncmVlbWVudCwKbmVpZ2hib3Igc3RkIC0tIHRvIHNldCBhbHBoYSBwZXIgcm93LiBIaWdoZXIgY29uZmlkZW5jZSAtPiBhbHBoYQpjbG9zZXIgdG8gMS4gTG93ZXIgY29uZmlkZW5jZSAtPiBhbHBoYSBjbG9zZXIgdG8gMC4KClRoaXMgbW9kdWxlIGlzIGEgc3RhbmQtYWxvbmUgYXBwbGllZCB0byBBTlkgT09GIHByZWRpY3Rpb24gYXJyYXkKKHY5LCB2OCwgc3RhY2tlciBvdXRwdXQpLiBJdCBwYWlycyBjbGVhbmx5IHdpdGggdGhlIG1ldGEtc3RhY2tlci4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCBudW1weSBhcyBucAoKCmRlZiBjb25zdGFudF9zaHJpbmthZ2UoCiAgICBkZWx0YV9wcmVkOiBucC5uZGFycmF5LAogICAgKiwKICAgIGFscGhhOiBmbG9hdCA9IDAuNywKKSAtPiBucC5uZGFycmF5OgogICAgIiIiTXVsdGlwbGljYXRpdmUgc2hyaW5rYWdlIHRvd2FyZCB6ZXJvIChpLmUuLCB0b3dhcmQgdGhlIGFuY2hvcikuCgogICAgUGFyYW1ldGVycwogICAgLS0tLS0tLS0tLQogICAgZGVsdGFfcHJlZCA6IChOLCkgbnAubmRhcnJheQogICAgICAgIE1vZGVsJ3MgcHJlZGljdGVkIGRlbHRhID0gVFZUIC0gbGFzdF9rbm93bl9UVlQuCiAgICBhbHBoYSA6IGZsb2F0CiAgICAgICAgU2hyaW5rYWdlIGZhY3Rvci4gMS4wID0gbm8gc2hyaW5rYWdlOyAwLjAgPSBwcmVkaWN0IGFuY2hvciBmb3IgYWxsLgogICAgICAgIFJlY29tbWVuZGVkIHN0YXJ0aW5nIHZhbHVlIDAuNyAoYXVkaXQgb24gZnVsbCB2OSBPT0YpLgoKICAgIFJldHVybnMKICAgIC0tLS0tLS0KICAgIHNocnVuayA6IChOLCkgbnAubmRhcnJheQogICAgICAgIGFscGhhICogZGVsdGFfcHJlZC4gVG8gcmVjb3ZlciBhYnNvbHV0ZSBUVlQsIGFkZCBsYXN0X2tub3duX1RWVC4KICAgICIiIgogICAgcmV0dXJuIGFscGhhICogZGVsdGFfcHJlZAoKCmRlZiBoYXJkX2NhcCgKICAgIGRlbHRhX3ByZWQ6IG5wLm5kYXJyYXksCiAgICAqLAogICAgYmFuZDogZmxvYXQgPSAzMC4wLAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJIYXJkLWNhcCBwcmVkaWN0ZWQgZGVsdGEgdG8gWy1iYW5kLCArYmFuZF0gKGluIGZ0KS4KCiAgICBQb3B1bGF0aW9uIHA5NSBvZiB8ZXZhbF9vZmZzZXRfZnJvbV9hbmNob3J8IGlzIH4yMCBmdCwgcDk5IH4zOCBmdC4KICAgIENhcHBpbmcgYXQgMzAgZnQgcHJlc2VydmVzIHJlYWwgbW90aW9uIGluIDk5JSBvZiB0eXBpY2FsIHdlbGxzIHdoaWxlCiAgICBjaG9wcGluZyB0aGUgY2F0YXN0cm9waGljIHRhaWwuCiAgICAiIiIKICAgIHJldHVybiBucC5jbGlwKGRlbHRhX3ByZWQsIC1iYW5kLCBiYW5kKQoKCmRlZiBnYXRlZF9zaHJpbmthZ2UoCiAgICBkZWx0YV9wcmVkOiBucC5uZGFycmF5LAogICAgY29uZmlkZW5jZTogbnAubmRhcnJheSwKICAgICosCiAgICBhbHBoYV9taW46IGZsb2F0ID0gMC40LAogICAgYWxwaGFfbWF4OiBmbG9hdCA9IDEuMCwKICAgIGNvbmZpZGVuY2VfY2xpcDogdHVwbGVbZmxvYXQsIGZsb2F0XSB8IE5vbmUgPSBOb25lLAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJQZXItcm93IHNocmlua2FnZSB3aXRoIGFscGhhIG1vZHVsYXRlZCBieSBhIGNvbmZpZGVuY2Ugc2lnbmFsLgoKICAgIGNvbmZpZGVuY2UgXGluIFswLCAxXTogMCA9IHRvdGFsbHkgdW50cnVzdGVkIChjb2xsYXBzZSB0byBhbmNob3IpLAogICAgMSA9IGZ1bGx5IHRydXN0ZWQgKG5vIHNocmlua2FnZSkuIFRoZSBtYXBwaW5nIGlzIGxpbmVhcjoKICAgICAgICBhbHBoYSA9IGFscGhhX21pbiArIChhbHBoYV9tYXggLSBhbHBoYV9taW4pICogY29uZmlkZW5jZQogICAgc28gYSByb3cgd2l0aCBjb25maWRlbmNlPTAgZ2V0cyBhbHBoYT1hbHBoYV9taW4gKG1heGltdW0gc2hyaW5rYWdlKS4KICAgICIiIgogICAgYyA9IG5wLmFzYXJyYXkoY29uZmlkZW5jZSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIGNvbmZpZGVuY2VfY2xpcCBpcyBub3QgTm9uZToKICAgICAgICBsbywgaGkgPSBjb25maWRlbmNlX2NsaXAKICAgICAgICBjID0gKGMgLSBsbykgLyBtYXgoaGkgLSBsbywgMWUtMTIpCiAgICBjID0gbnAuY2xpcChjLCAwLjAsIDEuMCkKICAgIGFscGhhID0gYWxwaGFfbWluICsgKGFscGhhX21heCAtIGFscGhhX21pbikgKiBjCiAgICByZXR1cm4gYWxwaGEgKiBkZWx0YV9wcmVkCgoKQGRhdGFjbGFzcwpjbGFzcyBTaHJpbmthZ2VSZXBvcnQ6CiAgICBvdmVyYWxsX3Jtc2U6IGZsb2F0CiAgICBvdmVyYWxsX21hZTogZmxvYXQKICAgIG92ZXJhbGxfYmlhczogZmxvYXQKICAgIG1lZGlhbl93ZWxsX3Jtc2U6IGZsb2F0CiAgICBwOTBfd2VsbF9ybXNlOiBmbG9hdAogICAgbWF4X3dlbGxfcm1zZTogZmxvYXQKCgpkZWYgZXZhbHVhdGVfc2hyaW5rYWdlKAogICAgZGVsdGFfcHJlZDogbnAubmRhcnJheSwKICAgIHRhcmdldDogbnAubmRhcnJheSwKICAgIHdlbGxfaWRzOiBucC5uZGFycmF5LAopIC0+IFNocmlua2FnZVJlcG9ydDoKICAgICIiIlNjb3JlIGEgc2hydW5rIHByZWRpY3Rpb24gb24gdGhlIE9PRiB0YXJnZXQuIiIiCiAgICBlcnIgPSBucC5hc2FycmF5KGRlbHRhX3ByZWQsIGR0eXBlPW5wLmZsb2F0NjQpIC0gbnAuYXNhcnJheSh0YXJnZXQsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICB3ZWxsX2lkcyA9IG5wLmFzYXJyYXkod2VsbF9pZHMpCiAgICBybXNlID0gZmxvYXQobnAuc3FydChucC5tZWFuKGVyciAqIGVycikpKQogICAgbWFlID0gZmxvYXQobnAubWVhbihucC5hYnMoZXJyKSkpCiAgICBiaWFzID0gZmxvYXQobnAubWVhbihlcnIpKQoKICAgIHdlbGxfdW5pcXVlID0gbnAudW5pcXVlKHdlbGxfaWRzKQogICAgd2VsbF9ybXNlID0gbnAuemVyb3Mod2VsbF91bmlxdWUuc2l6ZSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGZvciBpLCB3IGluIGVudW1lcmF0ZSh3ZWxsX3VuaXF1ZSk6CiAgICAgICAgbWFzayA9IHdlbGxfaWRzID09IHcKICAgICAgICBlID0gZXJyW21hc2tdCiAgICAgICAgd2VsbF9ybXNlW2ldID0gZmxvYXQobnAuc3FydChucC5tZWFuKGUgKiBlKSkpIGlmIGUuc2l6ZSBlbHNlIGZsb2F0KCJuYW4iKQogICAgcmV0dXJuIFNocmlua2FnZVJlcG9ydCgKICAgICAgICBvdmVyYWxsX3Jtc2U9cm1zZSwKICAgICAgICBvdmVyYWxsX21hZT1tYWUsCiAgICAgICAgb3ZlcmFsbF9iaWFzPWJpYXMsCiAgICAgICAgbWVkaWFuX3dlbGxfcm1zZT1mbG9hdChucC5tZWRpYW4od2VsbF9ybXNlKSksCiAgICAgICAgcDkwX3dlbGxfcm1zZT1mbG9hdChucC5wZXJjZW50aWxlKHdlbGxfcm1zZSwgOTApKSwKICAgICAgICBtYXhfd2VsbF9ybXNlPWZsb2F0KHdlbGxfcm1zZS5tYXgoKSksCiAgICApCg==",
    "triple_signal_pf.py": "...",
}
for _name, _payload in _MODULES.items():
    with open(os.path.join(SRC_DIR, _name), "wb") as _f:
        _f.write(base64.b64decode(_payload))

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# 2) Data root
INPUT_ROOT = "/kaggle/input"
DATA_ROOT = None
if os.path.isdir(INPUT_ROOT):
    for root, dirs, _files in os.walk(INPUT_ROOT):
        depth = root.replace(INPUT_ROOT, "").count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue
        if "test" in dirs and "train" in dirs:
            DATA_ROOT = root
            logger.info("Found competition data at %s", DATA_ROOT)
            break
if DATA_ROOT is None:
    raise SystemExit("FATAL: could not locate competition test/+train/ directories")

TRAIN_DIR = Path(DATA_ROOT) / "train"
TEST_DIR = Path(DATA_ROOT) / "test"

# 3) Imports
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception as _e:
    HAS_XGB = False
    logger.warning("XGBoost unavailable: %s", _e)

from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

from feature_builder import (
    FORMATIONS,
    FormationPlaneKNN,
    RowKNN,
    MLPAnccImputer,
    AnisoFormationImputer,
    build_dataset,
)
from anchor_shrinkage import constant_shrinkage, hard_cap
from triple_signal_pf import run_pfs_for_wells

# 4) Config
PRIMARY_FORMATION = "ANCC"
N_SPLITS = 5
SPLIT_SEED = 42
LGB_SEEDS = [42, 7, 123, 999, 31337]
ENABLE_BEAM = True
EWM_SPAN = 4.0
USE_GPU = True

SHRINK_ALPHA = 1.0
HARD_CAP_BAND = 40.0

MLP_NUM_FREQS = 8
MLP_HIDDEN = 256
MLP_EPOCHS = 12
MLP_ROWS_PER_EPOCH = 500_000
MLP_SEEDS = [42, 7, 123]

ANISO_KERNEL = "exponential"
ANISO_K = 20
ANISO_RANGE_SCALE = 1.0

PF_N_PARTICLES = 500
PF_SEED = 42

OUTPUT = Path("/kaggle/working/submission.csv")
OOF_OUT = Path("/kaggle/working/oof.csv")

# 5) Spatial imputers
train_paths = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
test_paths = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
logger.info("train paths=%d test paths=%d", len(train_paths), len(test_paths))

logger.info("Building plane-fit imputer ...")
formation_imputer = FormationPlaneKNN.fit(train_paths, formations=FORMATIONS)

logger.info("Building row-level KNN imputer ...")
row_imputer = RowKNN.fit(train_paths, formations=FORMATIONS)

logger.info("Building anisotropic-%s kriging imputer ...", ANISO_KERNEL)
aniso_imputer = AnisoFormationImputer.fit(
    train_paths, formations=FORMATIONS,
    kernel=ANISO_KERNEL, range_scale=ANISO_RANGE_SCALE, k=ANISO_K,
)

logger.info("Training %d-seed MLP ensemble ...", len(MLP_SEEDS))
mlp_imputer = MLPAnccImputer.fit(
    train_paths, formations=FORMATIONS,
    num_freqs=MLP_NUM_FREQS, hidden=MLP_HIDDEN,
    epochs=MLP_EPOCHS, rows_per_epoch=MLP_ROWS_PER_EPOCH,
    seeds=MLP_SEEDS, verbose=False,
)


# 6) Particle filters in parallel (across wells)
def _load_pl(path):
    return pl.read_csv(str(path), infer_schema_length=2000,
                       null_values=["", "NA", "NaN", "nan", "null"],
                       truncate_ragged_lines=True)


def _build_pf_input(paths_dir):
    well_dfs = {}
    typewell_dfs = {}
    for p in sorted(paths_dir.glob("*__horizontal_well.csv")):
        wid = p.stem.replace("__horizontal_well", "")
        tw_path = paths_dir / f"{wid}__typewell.csv"
        if not tw_path.exists():
            continue
        well_dfs[wid] = _load_pl(p)
        typewell_dfs[wid] = _load_pl(tw_path)
    return well_dfs, typewell_dfs


logger.info("Running TVT-PF + ANCC-PF on train wells (parallel) ...")
t0 = time.perf_counter()
train_well_dfs, train_typewell_dfs = _build_pf_input(TRAIN_DIR)
train_pf_results = run_pfs_for_wells(
    train_well_dfs, train_typewell_dfs,
    n_workers=-1, n_particles=PF_N_PARTICLES, seed=PF_SEED,
)
logger.info("  train PF done: %d wells in %.1fs",
            len(train_pf_results), time.perf_counter() - t0)

logger.info("Running TVT-PF + ANCC-PF on test wells (parallel) ...")
t0 = time.perf_counter()
test_well_dfs, test_typewell_dfs = _build_pf_input(TEST_DIR)
test_pf_results = run_pfs_for_wells(
    test_well_dfs, test_typewell_dfs,
    n_workers=-1, n_particles=PF_N_PARTICLES, seed=PF_SEED,
)
logger.info("  test PF done: %d wells in %.1fs",
            len(test_pf_results), time.perf_counter() - t0)
del train_well_dfs, train_typewell_dfs, test_well_dfs, test_typewell_dfs

# 7) Features
logger.info("Building train features ...")
train_df = build_dataset(
    train_paths, formation_imputer, row_imputer,
    is_train=True,
    mlp_imputer=mlp_imputer, aniso_imputer=aniso_imputer,
    pf_results=train_pf_results,
    primary_formation=PRIMARY_FORMATION,
    formations=FORMATIONS, enable_beam=ENABLE_BEAM, label="train",
)
logger.info("  train shape: %s", train_df.shape)

logger.info("Building test features ...")
test_df = build_dataset(
    test_paths, formation_imputer, row_imputer,
    is_train=False,
    mlp_imputer=mlp_imputer, aniso_imputer=aniso_imputer,
    pf_results=test_pf_results,
    primary_formation=PRIMARY_FORMATION,
    formations=FORMATIONS, enable_beam=ENABLE_BEAM, label="test",
)
logger.info("  test shape: %s", test_df.shape)

if train_df.empty or test_df.empty:
    raise SystemExit("FATAL: empty feature matrix")

feature_cols = [c for c in train_df.columns if c not in {"well", "prediction_id", "target"}]
pf_cols = [c for c in feature_cols if c.startswith("pf_") or c.startswith("gr_tw_off_") or c == "tw_gr_at_pf" or c == "gr_minus_tw_at_pf"]
logger.info("  #features: %d (PF features: %d)", len(feature_cols), len(pf_cols))

# 8) Splits
gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SPLIT_SEED)
splits = list(gkf.split(train_df, train_df["target"], groups=train_df["well"]))

# 9) Models
LGB_BASE = dict(
    boosting_type="gbdt", learning_rate=0.06, num_leaves=89,
    min_child_samples=10, min_child_weight=0.5,
    n_estimators=5000, n_jobs=-1,
    reg_alpha=2.03, reg_lambda=87.28,
    subsample=0.645, subsample_freq=1, colsample_bytree=0.821,
    objective="regression", metric="rmse", verbose=-1,
)
if USE_GPU:
    LGB_BASE.update(device_type="gpu", gpu_use_dp=False, max_bin=255)


def train_lgb(seed):
    logger.info("LGB seed=%d", seed)
    params = dict(LGB_BASE); params["random_state"] = seed
    oof = np.zeros(len(train_df), dtype=np.float32)
    test_pred = np.zeros(len(test_df), dtype=np.float32)
    for fold, (tr, va) in enumerate(splits):
        dtr = lgb.Dataset(train_df.iloc[tr][feature_cols], label=train_df.iloc[tr]["target"])
        dva = lgb.Dataset(train_df.iloc[va][feature_cols], label=train_df.iloc[va]["target"], reference=dtr)
        m = lgb.train(params, dtr, valid_sets=[dva],
                      num_boost_round=params["n_estimators"],
                      callbacks=[lgb.early_stopping(125, verbose=False),
                                 lgb.log_evaluation(period=500)])
        oof[va] = m.predict(train_df.iloc[va][feature_cols], num_iteration=m.best_iteration).astype(np.float32)
        rmse = float(np.sqrt(np.mean((oof[va] - train_df.iloc[va]["target"].values) ** 2)))
        logger.info("  fold %d: rmse=%.4f best_iter=%d", fold, rmse, m.best_iteration)
        test_pred += m.predict(test_df[feature_cols], num_iteration=m.best_iteration).astype(np.float32) / N_SPLITS
    overall = float(np.sqrt(np.mean((oof - train_df["target"].values) ** 2)))
    logger.info("LGB seed=%d: OOF rmse=%.4f", seed, overall)
    return oof, test_pred, overall


XGB_BASE = dict(
    objective="reg:squarederror", eval_metric="rmse",
    learning_rate=0.06, max_depth=8, min_child_weight=10,
    subsample=0.7, colsample_bytree=0.85,
    reg_alpha=1.0, reg_lambda=20.0,
    tree_method="hist", n_jobs=-1,
)
if USE_GPU:
    XGB_BASE.update(device="cuda")


def train_xgb(seed):
    if not HAS_XGB:
        return None, None, None
    logger.info("XGB seed=%d", seed)
    params = dict(XGB_BASE); params["seed"] = seed
    oof = np.zeros(len(train_df), dtype=np.float32)
    test_pred = np.zeros(len(test_df), dtype=np.float32)
    for fold, (tr, va) in enumerate(splits):
        dtr = xgb.DMatrix(train_df.iloc[tr][feature_cols].values, label=train_df.iloc[tr]["target"].values)
        dva = xgb.DMatrix(train_df.iloc[va][feature_cols].values, label=train_df.iloc[va]["target"].values)
        m = xgb.train(params, dtr, num_boost_round=5000,
                      evals=[(dva, "val")], early_stopping_rounds=125, verbose_eval=500)
        oof[va] = m.predict(dva, iteration_range=(0, m.best_iteration + 1)).astype(np.float32)
        rmse = float(np.sqrt(np.mean((oof[va] - train_df.iloc[va]["target"].values) ** 2)))
        logger.info("  fold %d: rmse=%.4f best_iter=%d", fold, rmse, m.best_iteration)
        dte = xgb.DMatrix(test_df[feature_cols].values)
        test_pred += m.predict(dte, iteration_range=(0, m.best_iteration + 1)).astype(np.float32) / N_SPLITS
    overall = float(np.sqrt(np.mean((oof - train_df["target"].values) ** 2)))
    logger.info("XGB seed=%d: OOF rmse=%.4f", seed, overall)
    return oof, test_pred, overall


results = {}
for seed in LGB_SEEDS:
    oof, tp, score = train_lgb(seed)
    results[f"lgb_{seed}"] = {"oof": oof, "test": tp, "rmse": score}

if HAS_XGB:
    oof_xgb, test_xgb, rmse_xgb = train_xgb(42)
    if oof_xgb is not None:
        results["xgb_42"] = {"oof": oof_xgb, "test": test_xgb, "rmse": rmse_xgb}

# 10) Ensemble
oof_avg = np.mean([r["oof"] for r in results.values()], axis=0)
test_avg = np.mean([r["test"] for r in results.values()], axis=0)
rmse_avg = float(np.sqrt(np.mean((oof_avg - train_df["target"].values) ** 2)))

stack_X = np.column_stack([r["oof"] for r in results.values()])
ridge = Ridge(alpha=1.0, fit_intercept=False, positive=True)
ridge.fit(stack_X, train_df["target"].values)
stack_oof = ridge.predict(stack_X)
rmse_stack = float(np.sqrt(np.mean((stack_oof - train_df["target"].values) ** 2)))
weights = ridge.coef_ / max(ridge.coef_.sum(), 1e-9)
logger.info("simple avg=%.4f  ridge=%.4f  weights=%s", rmse_avg, rmse_stack,
            {k: float(round(w, 3)) for k, w in zip(results.keys(), weights)})
stack_test = ridge.predict(np.column_stack([r["test"] for r in results.values()]))

if rmse_avg <= rmse_stack:
    final_test = test_avg; final_oof = oof_avg; final_rmse = rmse_avg
else:
    final_test = stack_test; final_oof = stack_oof; final_rmse = rmse_stack
logger.info("Final raw OOF rmse: %.4f", final_rmse)

# 11) Anchor-shrinkage post-process
shrunk_delta = constant_shrinkage(final_test.astype(np.float64), alpha=SHRINK_ALPHA)
shrunk_delta = hard_cap(shrunk_delta, band=HARD_CAP_BAND)
shrunk_oof = constant_shrinkage(final_oof.astype(np.float64), alpha=SHRINK_ALPHA)
shrunk_oof = hard_cap(shrunk_oof, band=HARD_CAP_BAND)
shrunk_oof_rmse = float(np.sqrt(np.mean((shrunk_oof - train_df["target"].values) ** 2)))
logger.info("Post-shrink OOF rmse: %.4f (alpha=%.2f, band=%.1f ft)",
            shrunk_oof_rmse, SHRINK_ALPHA, HARD_CAP_BAND)

# 12) Reconstruct + EWM
test_anchor = test_df["last_known_tvt"].to_numpy(dtype=np.float64)
test_tvt = test_anchor + shrunk_delta

submission = pd.DataFrame({
    "well": test_df["well"].values,
    "row_idx": test_df["row_idx"].astype(np.int32).values,
    "id": test_df["prediction_id"].values,
    "tvt": test_tvt,
}).sort_values(["well", "row_idx"]).reset_index(drop=True)


def _apply_ewm(group):
    g = group.copy()
    g["tvt"] = g["tvt"].ewm(span=EWM_SPAN, adjust=False).mean()
    return g


submission = submission.groupby("well", group_keys=False).apply(_apply_ewm)

submission_out = submission[["id", "tvt"]].copy()
if submission_out["tvt"].isna().any():
    bad = submission_out["tvt"].isna()
    submission_out.loc[bad, "tvt"] = test_anchor[bad.to_numpy()]
if not np.isfinite(submission_out["tvt"]).all():
    median_tvt = float(np.median(test_anchor[np.isfinite(test_anchor)]))
    bad = ~np.isfinite(submission_out["tvt"])
    submission_out.loc[bad, "tvt"] = median_tvt

submission_out.to_csv(OUTPUT, index=False)
logger.info("Wrote %s (%d rows)", OUTPUT, len(submission_out))
print(f"Submission: {len(submission_out)} rows, {submission_out['id'].nunique()} unique ids")
print(submission_out["tvt"].describe())
print(f"Final raw OOF rmse:    {final_rmse:.4f}")
print(f"Final shrunk OOF rmse: {shrunk_oof_rmse:.4f}")
